In [108]:
# testing hysteresis parameter calculation? 
import numpy as np
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates


storm_directory = 'data/constituents'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('data/shear_stress/average_total_shear_stress_corrected.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('data/sonde_data/sonde_down_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('data/sonde_data/sonde_up_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')

Process and Merge Data

In [109]:
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

In [110]:
# join shear stress data and the matching sonde data with the storm data
merged_storms = {}

for storm_name, storm_df in storms.items():
    if "down" in storm_name.lower():
        sonde_df = sonde_downstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    elif "up" in storm_name.lower():
        sonde_df = sonde_upstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    else:
        continue

    merged_storm_df = storm_df.join(shear_stress, how="left")
    merged_storm_df = merged_storm_df.join(sonde_df, how="left")
    merged_storms[storm_name] = merged_storm_df

# also join the sonde data with the shear stress data for the entire record 
sonde_downstream = sonde_downstream.join(shear_stress, how="left")
sonde_upstream = sonde_upstream.join(shear_stress, how="left")

In [111]:
merged_storms['st5_down']

,SS (uL/L),SSC (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L),LAB ID,Unnamed: 14,shear_stress,Turbidity FNU,fDOM RFU
Date_Time,,,,,,,,,,
2023-08-13 18:30:00,0.000,13.330000,0.0000,0.090,3.451,NaN,NaN,63.445700,10.868000,6.4620
2023-08-13 18:45:00,210.458,76.153846,0.0683,7.778,4.473,456.0,NaN,66.830985,13.716000,7.4420
2023-08-13 18:58:00,116.066,41.200000,0.0961,4.535,3.936,457.0,NaN,67.974764,16.154400,9.5616
2023-08-13 19:02:00,113.258,39.600000,0.0968,5.889,3.943,458.0,NaN,67.931246,14.494400,9.6416
2023-08-13 19:10:00,81.150,27.600000,0.0906,4.804,4.039,459.0,`,67.053310,11.944000,9.6900
2023-08-13 21:02:00,56.150,20.833333,0.0392,2.921,2.786,460.0,NaN,68.396623,7.864267,7.7592


Hysteresis index calculation functions

In [112]:
def normalize_storm_df(df, tau_col, constituent_cols, cols_to_normalize=None):
    out = df.copy()

    if cols_to_normalize is None:
        cols_to_normalize = [tau_col] + [c for c in constituent_cols if c in out.columns]

    for col in cols_to_normalize:
        if col not in out.columns:
            continue

        series = out[col].astype(float)
        smin = series.min(skipna=True)
        smax = series.max(skipna=True)

        if pd.isna(smin) or pd.isna(smax) or smax == smin:
            out[col] = np.nan
        else:
            out[col] = (series - smin) / (smax - smin)

    return out

def split_hydrograph(df, tau_col, constituent_col):
    ts = df[[tau_col, constituent_col]].dropna().sort_index()
    if ts.empty:
        return None, None, None

    peak_pos = ts[tau_col].to_numpy().argmax()
    peak_label = ts.index[peak_pos]

    rise = ts.iloc[:peak_pos + 1][[tau_col, constituent_col]]
    fall = ts.iloc[peak_pos:][[tau_col, constituent_col]]  # exclude the peak point from the falling limb
    return rise, fall, peak_label

def interpolate_time_at_tau(tau_target, limb_df, tau_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col]].dropna().sort_index()
    if len(limb) == 0:
        return None
    if len(limb) == 1:
        return limb.index[0]

    tau = limb[tau_col].to_numpy(dtype=float)
    t = limb.index.view("int64")

    exact = np.where(tau == tau_target)[0]
    if exact.size > 0:
        return limb.index[exact[0]]

    diffs = tau - tau_target
    crossings = np.where(diffs[:-1] * diffs[1:] <= 0)[0]

    if crossings.size > 0:
        i = crossings[0]
        tau0, tau1 = tau[i], tau[i + 1]
        t0, t1 = t[i], t[i + 1]
    else:
        if not allow_extrapolate:
            return None

        n = min(fit_points, len(tau))
        if n < 2:
            return limb.index[-1]

        if tau_target < np.nanmin(tau):
            tau_fit = tau[:n]
            t_fit = t[:n]
        else:
            tau_fit = tau[-n:]
            t_fit = t[-n:]

        if np.all(tau_fit == tau_fit[0]):
            return pd.to_datetime(int(t_fit[0]))

        slope, intercept = np.polyfit(tau_fit, t_fit, 1)
        t_target = slope * tau_target + intercept
        return pd.to_datetime(int(t_target))

    if tau1 == tau0:
        return pd.to_datetime(int(t0))

    t_target = t0 + (t1 - t0) * ((tau_target - tau0) / (tau1 - tau0))
    return pd.to_datetime(int(t_target))

def get_time_bracket_points(t_target, limb_df, tau_col, constituent_col, allow_extrapolate=True, fit_points=4):
    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col, constituent_col]].dropna().sort_index()
    if len(limb) < 2:
        return None

    t = limb.index.view("int64")
    tau = limb[tau_col].to_numpy(dtype=float)
    c = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    extrapolated = False

    if t[0] <= t_target_int <= t[-1]:
        upper_idx = np.searchsorted(t, t_target_int, side="right")
        lower_idx = max(upper_idx - 1, 0)
        upper_idx = min(upper_idx, len(t) - 1)
    elif not allow_extrapolate:
        return None
    else:
        n = min(fit_points, len(t))
        if n < 2:
            return None

        extrapolated = True
        if t_target_int < t[0]:
            lower_idx, upper_idx = 0, n - 1
        else:
            lower_idx, upper_idx = len(t) - n, len(t) - 1

    return {
        "t0": pd.to_datetime(int(t[lower_idx])),
        "t1": pd.to_datetime(int(t[upper_idx])),
        "tau0": tau[lower_idx],
        "tau1": tau[upper_idx],
        "c0": c[lower_idx],
        "c1": c[upper_idx],
        "extrapolated": extrapolated,
    }

def interpolate_constituent_at_time(t_target, limb_df, constituent_col, allow_extrapolate=True):
    if limb_df is None or limb_df.empty:
        return np.nan

    limb = limb_df[[constituent_col]].dropna().sort_index()
    if len(limb) == 0:
        return np.nan
    if len(limb) == 1:
        return limb[constituent_col].iloc[0]

    t = limb.index.view("int64")
    y = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    if t[0] <= t_target_int <= t[-1]:
        return np.interp(t_target_int, t, y)

    if not allow_extrapolate:
        return np.nan

    if t_target_int < t[0]:
        t0, t1 = t[0], t[1]
        y0, y1 = y[0], y[1]
    else:
        t0, t1 = t[-2], t[-1]
        y0, y1 = y[-2], y[-1]

    if t1 == t0:
        return y0

    slope = (y1 - y0) / (t1 - t0)
    return y0 + slope * (t_target_int - t0)

    if limb_df is None or limb_df.empty:
        return None

    limb = limb_df[[tau_col, constituent_col]].dropna().sort_index()
    if len(limb) < 2:
        return None

    t = limb.index.view("int64")
    tau = limb[tau_col].to_numpy(dtype=float)
    c = limb[constituent_col].to_numpy(dtype=float)
    t_target_int = pd.Timestamp(t_target).value

    extrapolated = False

    if t[0] <= t_target_int <= t[-1]:
        upper_idx = np.searchsorted(t, t_target_int, side="right")
        lower_idx = max(upper_idx - 1, 0)
        upper_idx = min(upper_idx, len(t) - 1)
    elif not allow_extrapolate:
        return None
    elif t_target_int < t[0]:
        lower_idx, upper_idx = 0, 1
        extrapolated = True
    else:
        lower_idx, upper_idx = len(t) - 2, len(t) - 1
        extrapolated = True

    return {
        "t0": pd.to_datetime(int(t[lower_idx])),
        "t1": pd.to_datetime(int(t[upper_idx])),
        "tau0": tau[lower_idx],
        "tau1": tau[upper_idx],
        "c0": c[lower_idx],
        "c1": c[upper_idx],
        "extrapolated": extrapolated,
    }

def calculate_lawler_HI(df, tau_col, constituent_col, storm_name, k=0.5):
    rise, fall, peak_label = split_hydrograph(df, tau_col, constituent_col)
    if rise is None or fall is None or rise.empty or fall.empty:
        print(f"Not enough data points to split the hydrograph for {storm_name}")
        return None

    tau_min = df[tau_col].min()
    tau_max = df[tau_col].max()
    tau_mid = k * (tau_max - tau_min) + tau_min

    t_rise_mid = interpolate_time_at_tau(tau_mid, rise, tau_col, allow_extrapolate=True)
    t_fall_mid = interpolate_time_at_tau(tau_mid, fall, tau_col, allow_extrapolate=True)

    if t_rise_mid is None or t_fall_mid is None:
        print(f"Could not find target time for storm {storm_name}")
        return None

    c_rise = interpolate_constituent_at_time(t_rise_mid, rise, constituent_col, allow_extrapolate=True)
    c_fall = interpolate_constituent_at_time(t_fall_mid, fall, constituent_col, allow_extrapolate=True)

    if np.isnan(c_rise) or np.isnan(c_fall):
        print(f"Could not interpolate storm {storm_name}")
        return None

    rise_bracket = get_time_bracket_points(t_rise_mid, rise, tau_col, constituent_col)
    fall_bracket = get_time_bracket_points(t_fall_mid, fall, tau_col, constituent_col)

    if c_rise > c_fall:
        HI = 1 - (c_fall / c_rise)
    elif c_rise < c_fall:
        HI = -1 + (c_rise / c_fall)
    else:
        HI = 0

    return {
        "storm_name": storm_name,
        "HI": HI,
        "tau_mid": tau_mid,
        "c_rise": c_rise,
        "c_fall": c_fall,
        "rise": rise,
        "fall": fall,
        "tau_col": tau_col,
        "constituent_col": constituent_col,
        "k": k,
        "t_rise_mid": t_rise_mid,
        "t_fall_mid": t_fall_mid,
        "rise_bracket": rise_bracket,
        "fall_bracket": fall_bracket,
        "peak_label": peak_label,
    }

def plot_lawler_hysteresis(results, df, tau_col, constituent_col,
                           xlabel=r"Shear Stress ($\tau$)",
                           ylabel="Constituent",
                           out_dir="HI_calculations/lawler_plots",
                           save=True,
                           show=False):

    rise = results["rise"]
    fall = results["fall"]
    HI = results["HI"]
    tau_mid = results["tau_mid"]
    c_rise = results["c_rise"]
    c_fall = results["c_fall"]
    storm_name = results.get("storm_name", "")
    k = results.get("k", 0.5)
    ts_df = df[[tau_col, constituent_col]].dropna().sort_index()

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1.2, 1]})

    ax = axes[0]
    ax.plot(ts_df.index, ts_df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.axhline(tau_mid, linestyle="--", color="tab:blue", alpha=0.7, linewidth=1.5, label=fr"$\tau_{{mid}}$")
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(MaxNLocator(8))

    ax2 = ax.twinx()
    ax2.plot(ts_df.index, ts_df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    t_rise_mid = results.get("t_rise_mid")
    t_fall_mid = results.get("t_fall_mid")

    if t_rise_mid is not None:
        ax.axvline(t_rise_mid, color="forestgreen", linestyle="--", alpha=0.5)
        ax2.scatter(t_rise_mid, c_rise, s=80, color="forestgreen", edgecolor="black", zorder=6, marker="D")
    if t_fall_mid is not None:
        ax.axvline(t_fall_mid, color="firebrick", linestyle="--", alpha=0.5)
        ax2.scatter(t_fall_mid, c_fall, s=80, color="firebrick", edgecolor="black", zorder=6, marker="D")

    # Plot the event peak as a black star (time-series axes and twin)
    peak_label = results.get("peak_label")
    if peak_label is not None and peak_label in ts_df.index:
        tau_peak = ts_df.at[peak_label, tau_col]
        c_peak = ts_df.at[peak_label, constituent_col]
        ax.scatter(peak_label, tau_peak, marker="*", color="k", s=160, zorder=9)

    event_date = ts_df.index[0].strftime("%Y-%m-%d")
    ax.set_title(f"Event time series ({event_date})")
    ax.set_xlabel("Time (HH:MM)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    combined = pd.concat([rise, fall.iloc[1:]])
    ax.plot(combined[tau_col], combined[constituent_col], color="black", linewidth=2, alpha=0.6, zorder=1)
    ax.scatter(rise[tau_col], rise[constituent_col], color="forestgreen", label="Rising limb", zorder=4)
    ax.scatter(fall[tau_col], fall[constituent_col], color="firebrick", label="Falling limb", zorder=4)

    ax.axvline(tau_mid, linestyle="--", color="gray", linewidth=2, label=fr"$\tau_{{mid}}$ (k={k})", zorder=2)
    ax.scatter(tau_mid, c_rise, s=80, color="forestgreen", edgecolor="black", zorder=5, marker="D")
    ax.scatter(tau_mid, c_fall, s=80, color="firebrick", edgecolor="black", zorder=5, marker="D")

    # also mark peak on tau-vs-constituent panel
    if peak_label is not None and peak_label in ts_df.index:
        ax.scatter(tau_peak, c_peak, marker="*", color="k", s=160, zorder=9, edgecolor="black")

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title(f"Lawler HI = {HI:.2f}")

    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    fig.suptitle(main_title, fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_lawler.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)

Normalize the shear stress and constituents in each storm

In [113]:
constituents = ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]

for storm_name, storm_df in merged_storms.items():
    merged_storms[storm_name] = normalize_storm_df(
        storm_df,
        tau_col="shear_stress",
        constituent_cols=constituents
    )

Calculate constituents hysteresis 

In [114]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue
        result = calculate_lawler_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/lawler_hysteresis_results.csv', index=False)

Plot constituents

In [115]:
for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        result = calculate_lawler_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name
        )

        if result is not None:
            plot_lawler_hysteresis(result, storm_df, "shear_stress", constituent, 
                                xlabel="Normalized shear stress",
                                ylabel="Normalized constituent",
                                out_dir="HI_calculations/lawler_plots")

C:\Users\huck4481\AppData\Local\Temp\ipykernel_26996\3875751311.py:303: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
C:\Users\huck4481\AppData\Local\Temp\ipykernel_26996\3875751311.py:328: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  fig.tight_layout(rect=[0, 0, 1, 0.95])
C:\Users\huck4481\AppData\Local\Temp\ipykernel_26996\3875751311.py:333: UserWarning: AutoDateLocator was unable to pick an appropriate interval for this date range. It may be necessary to add an interval value to the AutoDateLocator's intervald dictionary. Defaulting to 30.
  fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight"

### Turbidity and fDOM hysteresis

Separating storm by date and time

In [116]:
# define storm windows once
storm_windows = {
    "st1_down": ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st1_up":   ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st2_down": ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st2_up":   ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st3_down": ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st3_up":   ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st4_down": ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st4_up":   ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st5_down": ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st5_up":   ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st6_down": ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st6_up":   ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st7_down": ("2023-09-14 13:00", "2023-09-15 13:00"),
    "st7_up":   ("2023-09-14 13:00", "2023-09-15 13:00")
}

# build event dictionary directly (no function)
sonde_events = {}
down = sonde_downstream.sort_index()   
up = sonde_upstream.sort_index()      

for storm_name, (start, end) in storm_windows.items():
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if "down" in storm_name.lower():
        src = down
    elif "up" in storm_name.lower():
        src = up
    else:
        continue
    sonde_events[storm_name] = src.loc[(src.index >= start) & (src.index <= end)].copy()

Normalize shear stress and sonde data

In [117]:
sonde_constituents = ["Turbidity FNU", "fDOM RFU"]

for storm_name, storm_df in sonde_events.items():
    sonde_events[storm_name] = normalize_storm_df(
        storm_df,
        tau_col="shear_stress",
        constituent_cols=sonde_constituents
    )

Calculate HI for sonde data

In [118]:
sonde_results = []

for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        result = calculate_lawler_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            sonde_results.append(result)

sonde_results = pd.DataFrame(sonde_results)
sonde_results.to_csv('HI_calculations/lawler_sonde_hysteresis_results.csv', index=False)

Not enough data points to split the hydrograph for st2_up
Not enough data points to split the hydrograph for st2_up
Not enough data points to split the hydrograph for st3_up
Not enough data points to split the hydrograph for st3_up
Not enough data points to split the hydrograph for st5_up
Not enough data points to split the hydrograph for st5_up
Not enough data points to split the hydrograph for st6_up
Not enough data points to split the hydrograph for st6_up
Not enough data points to split the hydrograph for st7_down
Not enough data points to split the hydrograph for st7_down


Plot sonde data

In [119]:
for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        result = calculate_lawler_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name
        )

        if result is not None:
            plot_lawler_hysteresis(result, storm_df, "shear_stress", constituent, out_dir="HI_calculations/lawler_plots/sonde")

Not enough data points to split the hydrograph for st2_up
Not enough data points to split the hydrograph for st2_up
Not enough data points to split the hydrograph for st3_up
Not enough data points to split the hydrograph for st3_up
Not enough data points to split the hydrograph for st5_up
Not enough data points to split the hydrograph for st5_up
Not enough data points to split the hydrograph for st6_up
Not enough data points to split the hydrograph for st6_up
Not enough data points to split the hydrograph for st7_down
Not enough data points to split the hydrograph for st7_down
